# 02 - Entrenamiento LoRA

Entrena dos adapters sobre `Qwen/Qwen2.5-Coder-1.5B-Instruct` que difieren en un
solo factor, el **rank**. Las hipotesis estan escritas antes de entrenar en
`resultados/configs_lora.md`.

Requiere GPU. En Colab: Entorno de ejecucion > Cambiar tipo de entorno > T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -r requirements.txt

## 1. El dato que entra al modelo

El prompt se importa de `solucion.py`, no se copia: entrenar con un formato y
generar con otro es la forma mas silenciosa de perder puntos.

La perdida se calcula solo sobre la consulta; el esquema y la pregunta van
enmascarados con `-100`.

In [ ]:
import json
import sys

sys.path.insert(0, '.')
import solucion
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(solucion.CONFIG['modelo_base'])
pares = [json.loads(l) for l in open('datos/entrenamiento.jsonl', encoding='utf-8')]
esquemas = json.load(open('datos/esquemas_show_create.json', encoding='utf-8'))

ejemplo = pares[0]
prompt = solucion.construir_prompt(tok, esquemas[ejemplo['base']], ejemplo['pregunta'])
print(prompt[-700:])
print('--- objetivo ---')
print(ejemplo['sql'])

largos = [len(tok(solucion.construir_prompt(tok, esquemas[p['base']], p['pregunta']))['input_ids'])
          for p in pares[:200]]
print('\ntokens del prompt: mediana %d, maximo %d' % (sorted(largos)[len(largos)//2], max(largos)))

## 2. Configuracion A: rank 8

In [ ]:
!python scripts/entrenar.py --etiqueta r8 --rank 8 --alpha 32

## 3. Configuracion B: rank 32

Unico cambio respecto de A. `alpha` se mantiene en 32 a proposito: si se
escalara con el rank cambiarian dos cosas a la vez.

In [ ]:
!python scripts/entrenar.py --etiqueta r32 --rank 32 --alpha 32

## 4. Curvas de loss

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for eje, etiqueta in zip(ejes, ['r8', 'r32']):
    d = pd.read_csv('resultados/loss_%s.csv' % etiqueta)
    tr = d.dropna(subset=['loss_entrenamiento'])
    ev = d.dropna(subset=['loss_validacion'])
    eje.plot(tr['paso'], tr['loss_entrenamiento'], label='entrenamiento')
    eje.plot(ev['paso'], ev['loss_validacion'], marker='o', label='dev')
    eje.set_title('LoRA %s' % etiqueta)
    eje.set_xlabel('paso')
    eje.legend()
ejes[0].set_ylabel('loss')
plt.tight_layout()
plt.savefig('resultados/curvas_loss.png', dpi=140)
plt.show()

In [ ]:
import json

import pandas as pd

fichas = [json.load(open('resultados/corrida_%s.json' % e, encoding='utf-8'))
          for e in ['r8', 'r32']]
display(pd.DataFrame(fichas).set_index('etiqueta').T)

## 5. Elegir el adapter que se entrega

Se evaluan los dos con `scripts/evaluar.py` y se copia el ganador a `adapter/`,
que es la carpeta que lee `config.json`. Ante un empate se entrega `r8`: menos
parametros y menos riesgo de sobreajuste a las plantillas del dataset.

In [ ]:
!python scripts/evaluar.py --etiqueta lora_r8  --adapter adapter_r8
!python scripts/evaluar.py --etiqueta lora_r32 --adapter adapter_r32

In [ ]:
import shutil

GANADOR = 'adapter_r8'  # ajustar segun la tabla de EX
shutil.rmtree('adapter', ignore_errors=True)
shutil.copytree(GANADOR, 'adapter')
print('adapter/ <-', GANADOR)
print(sorted(p.name for p in __import__('pathlib').Path('adapter').iterdir()))